In [ ]:
import datetime
import numpy as np
import cv2
from itertools import cycle
import pickle
import pathlib
import scipy.io
from matplotlib import pyplot as plt
import pandas as pd
from eye_tracking_system_tools.preprocessing.BlockSync_class import BlockSync
from eye_tracking_system_tools.preprocessing import utility_functions as uf
from matplotlib import rcParams
%matplotlib inline
plt.style.use('default')
rcParams['pdf.fonttype'] = 42  # Ensure fonts are embedded and editable
rcParams['ps.fonttype'] = 42  # Ensure compatibility with vector outputs


In [ ]:

# define a single block to work on
# this step creates block_collection - a list of BlockSync objects of interest
block_numbers = [7]
bad_blocks = [] #
experiment_path = r"Z:\Nimrod\experiments"
animal = 'PV_126'

block_collection = uf.block_generator(block_numbers=block_numbers,
                                      experiment_path=experiment_path,
                                      animal=animal,
                                      bad_blocks=bad_blocks,regev=True)
for block in block_collection:
    block.channeldict = None
    if block.animal_call == 'PV_208':
        block.channeldict={1: 'LED_driver',
                           7: 'L_eye_TTL',
                           2: 'Arena_TTL',
                           8: 'R_eye_TTL'}
# create a block_dict object for ease of access:
block_dict = {}
for b in block_collection:
    block_dict[str(b.block_num)] = b
block = block_collection[0]
       

In [ ]:

for block in block_collection: 
    block.left_eye_data = pd.read_csv(block.analysis_path / f'left_eye_data_degrees_raw_verified.csv')
    block.right_eye_data = pd.read_csv(block.analysis_path / 'right_eye_data_degrees_raw_verified.csv')


In [ ]:
def load_final_sync_df(block, filename=None, verbose=True):
    """
    Load a downstream-compatible final sync dataframe from disk and set:
      - block.final_sync_df
      - block.blocksync_df  (legacy compatibility)

    If `filename` is None, tries 'final_sync_df.csv' then 'blocksync_df.csv'
    inside `block.analysis_path`.

    Returns
    -------
    pd.DataFrame
    """
    # 1) pick a file
    ap = Path(block.analysis_path)
    candidates = [filename] if filename else ["final_sync_df.csv", "blocksync_df.csv"]
    path = None
    for name in candidates:
        p = ap / name
        if p.exists():
            path = p
            break
    if path is None:
        raise FileNotFoundError(f"No sync file found. Tried: {', '.join(str(ap / n) for n in candidates)}")

    # 2) read & validate schema
    df = pd.read_csv(path)
    required = ['Arena_TTL','Arena_frame','L_eye_frame','R_eye_frame','L_values','R_values']
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{path.name} is missing required columns: {missing}")

    # 3) light coercions to keep downstream happy
    df = df.copy()
    df['Arena_TTL'] = df['Arena_TTL'].astype(float)

    # 4) set attributes
    block.final_sync_df = df
    block.blocksync_df = df  # some older code reads this
    if verbose:
        print(f"[OK] Loaded {path.name} → block.final_sync_df (rows={len(df):,})")

    return df
for block in block_collection:
    load_final_sync_df(block)

block.final_sync_df['ms_axis'] = block.final_sync_df['Arena_TTL'].values / (block.sample_rate / 1000)

In [ ]:
block.handle_eye_videos()
block.handle_arena_files()
block.calibrate_pixel_size(10)
if 'pupil_diameter' not in block.left_eye_data.columns:
    print(f'calculating pupil diameter for {block} ')
    block.left_eye_data['pupil_diameter_pixels'] = block.left_eye_data.major_ax
    block.right_eye_data['pupil_diameter_pixels'] = block.right_eye_data.major_ax
    block.left_eye_data['pupil_diameter'] = block.left_eye_data['pupil_diameter_pixels'] * block.L_pix_size
    block.right_eye_data['pupil_diameter'] = block.right_eye_data['pupil_diameter_pixels'] * block.R_pix_size

In [ ]:

def recenter_eye_angles_to_rest(
        eye_df: pd.DataFrame,
        phi_col: str = "k_phi",
        theta_col: str = "k_theta",
        inplace: bool = False,
        dropna: bool = True,
) -> Tuple[pd.DataFrame, Dict[str, float]]:
    """
    Recenter the eye's angular coordinates so that the median (resting position)
    of k_phi and k_theta becomes zero degrees.

    Parameters
    ----------
    eye_df : pd.DataFrame
        Eye dataframe containing k_phi and k_theta columns (can be *_eye_data_clean).
    phi_col, theta_col : str
        Column names for the vertical (phi) and horizontal (theta) gaze angles.
    inplace : bool
        If True, modifies the input DataFrame in place; otherwise returns a copy.
    dropna : bool
        Whether to ignore NaN values when computing medians (recommended True).

    Returns
    -------
    df_out : pd.DataFrame
        DataFrame with corrected columns (values centered around 0).
    offsets : dict
        {'phi_offset': float, 'theta_offset': float}
        The medians that were subtracted from the original data.

    Notes
    -----
    - New columns 'k_phi_recentered' and 'k_theta_recentered' are added
      (or the originals replaced if inplace=True).
    - This assumes your angular units are degrees.
    - The offsets correspond to the eye's resting orientation in the camera frame.
    """
    # --- validate ---
    for c in (phi_col, theta_col):
        if c not in eye_df.columns:
            raise ValueError(f"Column '{c}' not found in dataframe")

    # Select working DataFrame
    df = eye_df if inplace else eye_df.copy()

    # Convert to numeric, handle NaN
    phi_vals = pd.to_numeric(df[phi_col], errors="coerce")
    theta_vals = pd.to_numeric(df[theta_col], errors="coerce")

    # Compute medians (resting position)
    phi_med = float(np.nanmedian(phi_vals)) if dropna else float(np.median(phi_vals))
    theta_med = float(np.nanmedian(theta_vals)) if dropna else float(np.median(theta_vals))

    # Apply centering
    df[phi_col] = phi_vals - phi_med
    df[theta_col] = theta_vals - theta_med

    # Optionally store explicitly labeled recentered columns
    df[f"{phi_col}_recentered"] = df[phi_col]
    df[f"{theta_col}_recentered"] = df[theta_col]

    offsets = {"phi_offset": phi_med, "theta_offset": theta_med}
    return df, offsets


# Apply to the cleaned left eye data
df_left_clean = block.left_eye_data
df_left_centered, left_offsets = recenter_eye_angles_to_rest(df_left_clean)

# Same for right
df_right_clean = block.right_eye_data
df_right_centered, right_offsets = recenter_eye_angles_to_rest(df_right_clean)

# Store back on the block if desired
block.left_eye_data_centered = df_left_centered
block.right_eye_data_centered = df_right_centered

print("Left eye offsets:", left_offsets)
print("Right eye offsets:", right_offsets)

In [ ]:
from __future__ import annotations

import os
from pathlib import Path
from typing import Optional, Sequence, Dict, Tuple, Union, Literal

import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm


class MonotoneFrameReader:
    """
    Frame-exact reader for mostly-nondecreasing frame indices.
    Avoids CAP_PROP_POS_FRAMES random seeking issues with MP4/H264.
    """
    def __init__(self, path, label="video"):
        self.path = str(path)
        self.label = label
        self.cap = cv2.VideoCapture(self.path)
        if not self.cap.isOpened():
            raise RuntimeError(f"Cannot open {label}: {path}")
        self.cur_idx = -1
        self.cur_frame = None

    def close(self):
        try:
            self.cap.release()
        except Exception:
            pass

    def _reopen_and_seek(self, target_idx: int):
        self.close()
        self.cap = cv2.VideoCapture(self.path)
        if not self.cap.isOpened():
            raise RuntimeError(f"Cannot reopen {self.label}: {self.path}")
        self.cur_idx = -1
        self.cur_frame = None
        if target_idx > 0:
            for _ in range(target_idx):
                ok = self.cap.grab()
                if not ok:
                    return None

    def read_at(self, target_idx: Optional[int]):
        if target_idx is None or target_idx < 0:
            return None
        target_idx = int(target_idx)

        if target_idx == self.cur_idx and self.cur_frame is not None:
            return self.cur_frame

        if target_idx < self.cur_idx:
            self._reopen_and_seek(target_idx)

        while self.cur_idx < target_idx:
            ok, frame = self.cap.read()
            if not ok:
                return None
            self.cur_idx += 1
            self.cur_frame = frame

        return self.cur_frame


def export_block_synchronized_montage_video(
    block: object,
    start_ms: float,
    end_ms: float,
    out_path: Union[os.PathLike, str],
    *,
    fps: float = 60.0,

    arena_video: Optional[Union[int, str]] = None,
    arena_frame_cols: Sequence[str] = (
        "Arena_frame", "arena_frame", "arena_frames", "arena_frame_idx",
        "frame", "frame_idx", "video_frame", "arena_idx"
    ),
    arena_frame_shift: int = 0,

    L_eye_frame_cols: Sequence[str] = ("L_eye_frame", "left_eye_frame", "le_eye_frame", "L_frame", "le_frame", "L_eye_idx"),
    R_eye_frame_cols: Sequence[str] = ("R_eye_frame", "right_eye_frame", "re_eye_frame", "R_frame", "re_frame", "R_eye_idx"),

    eye_video_mode: Literal["auto", "raw", "dlc"] = "auto",
    dlc_name_hint: str = "DLC",

    top_banner_h: int = 60,
    trace_h: int = 220,
    trace_scale: float = 2.0,
    flip_eyes_vertical: bool = True,

    trace_window_ms: Optional[float] = None,
    normalize_traces: bool = False,

    # default traces stay the same; mapping changes to recentered columns
    trace_signals: Sequence[str] = ("pupil_diameter", "phi", "theta"),
    trace_col_map: Optional[Dict[str, str]] = None,

    # NEW: prefer centered dataframes for traces
    use_centered_eye_data: bool = True,
    centered_suffix: str = "_centered",

    disqualify_cols: Sequence[str] = ("center_x", "center_y"),
    show_disqualified_badge: bool = True,

    require_all_three: bool = True,

    codec: str = "mp4v",
    timestamp_precision_ms: int = 0,
    show_debug_prints: bool = True,
) -> Path:
    

    # -------------------------- helpers --------------------------
    def _require_attr(obj: object, name: str):
        if not hasattr(obj, name):
            raise AttributeError(f"block is missing required attribute '{name}'")
        return getattr(obj, name)

    def _get_attr_optional(obj: object, name: str, default=None):
        return getattr(obj, name, default)

    def _require_cols(df: pd.DataFrame, cols: Sequence[str], df_name: str):
        missing = [c for c in cols if c not in df.columns]
        if missing:
            raise ValueError(f"{df_name} is missing required columns: {missing}")

    def _pick_first_video(paths, name: str) -> Path:
        if paths is None:
            raise ValueError(f"block.{name} is None")
        if isinstance(paths, (str, os.PathLike)):
            p = Path(paths)
            if not p.exists():
                raise FileNotFoundError(p)
            return p
        if isinstance(paths, (list, tuple)) and len(paths) > 0:
            p = Path(paths[0])
            if not p.exists():
                raise FileNotFoundError(p)
            return p
        raise ValueError(f"block.{name} has no usable video path(s)")

    def _open_cap(p: Path, label: str) -> cv2.VideoCapture:
        cap = cv2.VideoCapture(str(p))
        if not cap.isOpened():
            raise RuntimeError(f"Cannot open {label} video: {p}")
        return cap

    def _resolve_col(df: pd.DataFrame, candidates: Sequence[str], what: str) -> str:
        for c in candidates:
            if c in df.columns:
                return c
        raise ValueError(f"final_sync_df has no recognizable {what} column. Tried: {list(candidates)}")

    def _safe_put_text(img, text, org, color, scale=0.6, thickness=2):
        x, y = org
        cv2.putText(img, text, (x + 1, y + 1), cv2.FONT_HERSHEY_SIMPLEX, scale, (0, 0, 0),
                    thickness + 2, cv2.LINE_AA)
        cv2.putText(img, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX, scale, color, thickness, cv2.LINE_AA)

    def _make_banner(W: int, title: str) -> np.ndarray:
        banner = np.zeros((top_banner_h, W, 3), dtype=np.uint8)
        (tw, _), _ = cv2.getTextSize(title, cv2.FONT_HERSHEY_SIMPLEX, 0.9, 2)
        x = max(12, (W - tw) // 2)
        _safe_put_text(banner, title, (x, 34), (255, 255, 255), scale=0.9, thickness=2)
        return banner

    def _clamp_idx(idx: Optional[int], cap: cv2.VideoCapture) -> Optional[int]:
        if idx is None:
            return None
        n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
        if n > 0:
            return int(np.clip(int(idx), 0, n - 1))
        return max(0, int(idx))

    def _resize_to_height(img: np.ndarray, target_h: int) -> np.ndarray:
        h, w = img.shape[:2]
        if h == target_h:
            return img
        new_w = max(1, int(round(w * (target_h / float(h)))))
        return cv2.resize(img, (new_w, target_h), interpolation=cv2.INTER_AREA)

    def _map_to_axis(vals: np.ndarray, lo: float, hi: float) -> np.ndarray:
        return (vals - lo) / (hi - lo + 1e-12)

    def _limits_minmax_std(x: np.ndarray) -> Tuple[float, float]:
        x = np.asarray(x, dtype=float)
        x = x[np.isfinite(x)]
        if x.size < 2:
            return (-1.0, 1.0)
        mn = float(np.min(x))
        mx = float(np.max(x))
        sd = float(np.std(x))
        if not np.isfinite(sd) or sd < 1e-12:
            sd = max(1.0, 0.05 * (mx - mn) if (mx - mn) > 0 else 1.0)
        lo = mn - sd
        hi = mx + sd
        if not np.isfinite(lo) or not np.isfinite(hi) or abs(hi - lo) < 1e-12:
            lo, hi = mn - 1.0, mx + 1.0
        if abs(hi - lo) < 1e-12:
            lo -= 1.0
            hi += 1.0
        return lo, hi

    def _blend_thickness_1p5(panel: np.ndarray, overlay: np.ndarray, p0, p1, color):
        # ~1.5px by blending 1px + (0.5 * 2px)
        cv2.line(panel, p0, p1, color, 1, cv2.LINE_AA)
        cv2.line(overlay, p0, p1, color, 2, cv2.LINE_AA)

    def _draw_trace_panel(W: int, t_ms: float, t_grid: np.ndarray,
                          Lsig: Dict[str, np.ndarray], Rsig: Dict[str, np.ndarray],
                          t0: float, t1: float, trace_h_eff: int) -> np.ndarray:
        panel = np.zeros((trace_h_eff, W, 3), dtype=np.uint8)

        if trace_window_ms is None:
            w0, w1 = t0, t1
        else:
            half = float(trace_window_ms) / 2.0
            w0, w1 = max(t0, t_ms - half), min(t1, t_ms + half)
            if w1 <= w0:
                w0, w1 = t0, t1

        idx = np.where((t_grid >= w0) & (t_grid <= w1))[0]
        if idx.size < 2:
            return panel

        tg = t_grid[idx]
        x = (tg - w0) / (w1 - w0 + 1e-12)
        xpix = (x * (W - 1)).astype(int)

        n_axes = len(trace_signals)
        pad_y = 12
        axis_h = max(60, (trace_h_eff - 2 * pad_y) // max(1, n_axes))
        axes = []
        for j, name in enumerate(trace_signals):
            y0 = pad_y + j * axis_h
            y1 = min(trace_h_eff - pad_y, y0 + axis_h) - 10
            axes.append((name, y0, y1))

        cursor_x = int(round((np.clip(t_ms, w0, w1) - w0) / (w1 - w0 + 1e-12) * (W - 1)))
        cv2.line(panel, (cursor_x, 0), (cursor_x, trace_h_eff - 1), (120, 120, 120), 1)

        _safe_put_text(panel, f"t = {t_ms:.{timestamp_precision_ms}f} ms", (12, 26),
                       (255, 255, 255), scale=0.7, thickness=2)

        # OpenCV BGR
        color_L = (255, 0, 0)   # blue
        color_R = (0, 0, 255)   # red

        for name, y0, y1 in axes:
            cv2.rectangle(panel, (0, y0), (W - 1, y1), (20, 20, 20), 1)
            L = Lsig.get(name, None)
            R = Rsig.get(name, None)
            if L is None or R is None:
                continue

            Lw = L[idx].astype(float)
            Rw = R[idx].astype(float)

            yy0, yy1 = int(y0 + 20), int(y1 - 10)
            Hax = max(2, yy1 - yy0)

            if normalize_traces:
                # per-eye normalization (kept from v6 behavior)
                Llo, Lhi = np.nanmin(Lw), np.nanmax(Lw)
                Rlo, Rhi = np.nanmin(Rw), np.nanmax(Rw)
                Ln = np.full_like(Lw, 0.5) if (not np.isfinite(Llo) or not np.isfinite(Lhi) or abs(Lhi - Llo) < 1e-12) else _map_to_axis(Lw, Llo, Lhi)
                Rn = np.full_like(Rw, 0.5) if (not np.isfinite(Rlo) or not np.isfinite(Rhi) or abs(Rhi - Rlo) < 1e-12) else _map_to_axis(Rw, Rlo, Rhi)
                label = f"{name} (norm)"
            else:
                # v6+ behavior: limits based on data in current window with ±1 std margin
                lo, hi = _limits_minmax_std(np.concatenate([Lw, Rw]))
                Ln = _map_to_axis(Lw, lo, hi)
                Rn = _map_to_axis(Rw, lo, hi)
                label = f"{name} [{lo:.2f},{hi:.2f}]"

            yL_f = yy0 + (1.0 - np.clip(Ln, 0.0, 1.0)) * (Hax - 1)
            yR_f = yy0 + (1.0 - np.clip(Rn, 0.0, 1.0)) * (Hax - 1)

            mL = np.isfinite(yL_f)
            mR = np.isfinite(yR_f)

            yL_i = yL_f.astype(np.int32, copy=False)
            yR_i = yR_f.astype(np.int32, copy=False)

            overlay = np.zeros_like(panel)

            for k in range(1, len(xpix)):
                if mL[k - 1] and mL[k]:
                    _blend_thickness_1p5(
                        panel, overlay,
                        (int(xpix[k - 1]), int(yL_i[k - 1])),
                        (int(xpix[k]), int(yL_i[k])),
                        color_L
                    )
                if mR[k - 1] and mR[k]:
                    _blend_thickness_1p5(
                        panel, overlay,
                        (int(xpix[k - 1]), int(yR_i[k - 1])),
                        (int(xpix[k]), int(yR_i[k])),
                        color_R
                    )

            mask = np.any(overlay != 0, axis=2)
            if np.any(mask):
                panel[mask] = (0.5 * panel[mask].astype(np.float32) + 0.5 * overlay[mask].astype(np.float32)).astype(np.uint8)

            _safe_put_text(panel, label, (12, y0 + 18), (200, 200, 200), scale=0.6, thickness=1)
            _safe_put_text(panel, "L", (W - 60, y0 + 18), color_L, scale=0.6, thickness=2)
            _safe_put_text(panel, "R", (W - 35, y0 + 18), color_R, scale=0.6, thickness=2)

        return panel

    def _choose_arena_video(arena_list: Sequence[str], choice: Optional[Union[int, str]]) -> Path:
        paths = [Path(p) for p in arena_list]
        if not paths:
            raise ValueError("block.arena_videos is empty.")
        if isinstance(choice, int):
            if choice < 0 or choice >= len(paths):
                raise ValueError(f"arena_video index {choice} out of range (0..{len(paths)-1})")
            return paths[int(choice)]
        if isinstance(choice, str) and choice.strip():
            key = choice.strip().lower()
            hits = [p for p in paths if key in p.name.lower()]
            if not hits:
                raise ValueError(f"arena_video='{choice}' did not match any arena video name.")
            hits.sort(key=lambda p: len(p.name))
            return hits[0]
        print("\nSelect arena video:")
        for i, p in enumerate(paths):
            print(f"  [{i}] {p.name}")
        raw = ""
        try:
            raw = input("Enter arena index (default 0): ").strip()
        except Exception:
            raw = ""
        idx = 0 if raw == "" else int(raw)
        if idx < 0 or idx >= len(paths):
            raise ValueError(f"arena index {idx} out of range.")
        return paths[idx]

    def _resolve_eye_video(raw_path: Path, mode: str) -> Path:
        raw_path = Path(raw_path)
        if mode == "raw":
            return raw_path

        folder = raw_path.parent
        if not folder.exists():
            if mode == "dlc":
                raise FileNotFoundError(f"Eye video folder not found: {folder}")
            return raw_path

        hint = dlc_name_hint.lower()
        dlc_candidates = [p for p in folder.glob("*.mp4") if hint in p.name.lower()]

        stem = raw_path.stem.lower()
        stem_hits = [p for p in dlc_candidates if stem in p.stem.lower()]
        candidates = stem_hits if stem_hits else dlc_candidates

        if candidates:
            candidates.sort(key=lambda p: p.name)
            return candidates[0]

        if mode == "dlc":
            raise FileNotFoundError(
                f"eye_video_mode='dlc' but no DLC mp4 found in {folder} (hint='{dlc_name_hint}')"
            )
        return raw_path

    # -------------------------- validation --------------------------
    start_ms = float(start_ms)
    end_ms = float(end_ms)
    if end_ms <= start_ms:
        raise ValueError(f"end_ms must be > start_ms (got start_ms={start_ms}, end_ms={end_ms})")

    fsync = _require_attr(block, "final_sync_df")
    if not isinstance(fsync, pd.DataFrame):
        raise ValueError("block.final_sync_df is not a pandas DataFrame.")

    arena_fcol = _resolve_col(fsync, arena_frame_cols, "arena frame")
    L_fcol = _resolve_col(fsync, L_eye_frame_cols, "LEFT eye frame")
    R_fcol = _resolve_col(fsync, R_eye_frame_cols, "RIGHT eye frame")

    # timebase for final_sync_df rows
    if "ms_axis" in fsync.columns:
        ms_all = fsync["ms_axis"].to_numpy(dtype=float)
    elif "Arena_TTL" in fsync.columns:
        sr = float(_get_attr_optional(block, "sample_rate", None) or block.get_sample_rate())
        ms_all = fsync["Arena_TTL"].to_numpy(dtype=float) / sr * 1000.0
    elif "oe_time_s" in fsync.columns:
        ms_all = fsync["oe_time_s"].to_numpy(dtype=float) * 1000.0
    else:
        raise ValueError("final_sync_df has no 'ms_axis' and no usable time column ('Arena_TTL' or 'oe_time_s').")

    mask = np.isfinite(ms_all) & (ms_all >= start_ms) & (ms_all <= end_ms)
    if not np.any(mask):
        raise ValueError("No final_sync_df rows fall inside [start_ms, end_ms]. Check timebase and units.")
    idx_rows = np.where(mask)[0]
    fs_win = fsync.iloc[idx_rows].copy()
    t_ms = ms_all[idx_rows].astype(float)

    # fps subsampling
    if len(t_ms) > 5:
        dt = np.median(np.diff(t_ms))
        fps_master = 1000.0 / dt if dt > 0 else float("nan")
        if np.isfinite(fps_master) and fps_master > 0:
            stride = int(round(fps_master / float(fps))) if float(fps) <= fps_master else 1
            stride = max(1, stride)
            if stride > 1:
                fs_win = fs_win.iloc[::stride].copy()
                t_ms = t_ms[::stride]
                if show_debug_prints:
                    print(f"[export] final_sync_df master fps≈{fps_master:.3f}; requested fps={fps:.3f}; using stride={stride} (effective fps≈{fps_master/stride:.3f})")
            else:
                if show_debug_prints:
                    print(f"[export] final_sync_df master fps≈{fps_master:.3f}; requested fps={fps:.3f}; using all rows")

    # frames
    A_raw = fs_win[arena_fcol].to_numpy(dtype=float)
    L_raw = fs_win[L_fcol].to_numpy(dtype=float)
    R_raw = fs_win[R_fcol].to_numpy(dtype=float)

    if require_all_three:
        ok = np.isfinite(A_raw) & np.isfinite(L_raw) & np.isfinite(R_raw)
        fs_win = fs_win.loc[ok].copy()
        t_ms = t_ms[ok]
        A_raw = A_raw[ok]; L_raw = L_raw[ok]; R_raw = R_raw[ok]
        if len(fs_win) < 5:
            raise RuntimeError("Too few valid rows after require_all_three filtering; set require_all_three=False or inspect frame columns.")

    if arena_frame_shift != 0:
        A_raw = A_raw + float(int(arena_frame_shift))

    A_frames = np.array([int(v) if np.isfinite(v) else -1 for v in A_raw], dtype=np.int64)
    L_frames = np.array([int(v) if np.isfinite(v) else -1 for v in L_raw], dtype=np.int64)
    R_frames = np.array([int(v) if np.isfinite(v) else -1 for v in R_raw], dtype=np.int64)

    # -------------------------- choose which eye dataframes to use for traces --------------------------
    left_df_raw: pd.DataFrame = _require_attr(block, "left_eye_data")
    right_df_raw: pd.DataFrame = _require_attr(block, "right_eye_data")
    _require_cols(left_df_raw, ["eye_frame"], "left_eye_data")
    _require_cols(right_df_raw, ["eye_frame"], "right_eye_data")

    left_centered = _get_attr_optional(block, "left_eye_data_centered", None)
    right_centered = _get_attr_optional(block, "right_eye_data_centered", None)

    if use_centered_eye_data:
        if left_centered is None or right_centered is None:
            raise AttributeError(
                "use_centered_eye_data=True but block.left_eye_data_centered / block.right_eye_data_centered not found."
            )
        if not isinstance(left_centered, pd.DataFrame) or not isinstance(right_centered, pd.DataFrame):
            raise TypeError("block.left_eye_data_centered / right_eye_data_centered must be pandas DataFrames.")
        left_df = left_centered
        right_df = right_centered
        _require_cols(left_df, ["eye_frame"], "left_eye_data_centered")
        _require_cols(right_df, ["eye_frame"], "right_eye_data_centered")
    else:
        left_df = left_df_raw
        right_df = right_df_raw

    # -------------------------- traces mapping (phi/theta -> recentered) --------------------------
    default_map = {
        "pupil_diameter": "pupil_diameter",
        "phi": "k_phi_recentered",
        "theta": "k_theta_recentered",
    }

    if trace_col_map is None:
        trace_col_map = default_map
    else:
        tmp = default_map.copy()
        tmp.update(trace_col_map)
        trace_col_map = tmp

    # Index eye dataframes by eye_frame for fast reindexing
    Ltab = left_df.drop_duplicates(subset=["eye_frame"], keep="first").set_index("eye_frame", drop=False)
    Rtab = right_df.drop_duplicates(subset=["eye_frame"], keep="first").set_index("eye_frame", drop=False)

    # Use the exact eye_frame numbers used for frame grabbing
    L_eye_idx = pd.Index(L_frames, name="eye_frame")
    R_eye_idx = pd.Index(R_frames, name="eye_frame")

    Lsig: Dict[str, np.ndarray] = {}
    Rsig: Dict[str, np.ndarray] = {}
    for sig in trace_signals:
        col = trace_col_map.get(sig, sig)
        if (col not in Ltab.columns) or (col not in Rtab.columns):
            Lsig[sig] = np.full_like(t_ms, np.nan, dtype=float)
            Rsig[sig] = np.full_like(t_ms, np.nan, dtype=float)
            continue
        Lsig[sig] = Ltab.reindex(L_eye_idx)[col].to_numpy(dtype=float)
        Rsig[sig] = Rtab.reindex(R_eye_idx)[col].to_numpy(dtype=float)

    # disqualification flags sampled by same mapping (note: uses whichever df is selected)
    def _disq_flags(tab: pd.DataFrame, eye_idx: pd.Index) -> np.ndarray:
        if not disqualify_cols:
            return np.zeros(len(eye_idx), dtype=bool)
        sub = tab.reindex(eye_idx)
        flags = np.zeros(len(eye_idx), dtype=bool)
        for c in disqualify_cols:
            if c in sub.columns:
                flags |= sub[c].isna().to_numpy()
        return flags

    disqL_flags = _disq_flags(Ltab, L_eye_idx)
    disqR_flags = _disq_flags(Rtab, R_eye_idx)

    # -------------------------- resolve chosen video files --------------------------
    rv_raw = _pick_first_video(_require_attr(block, "re_videos"), "re_videos")
    lv_raw = _pick_first_video(_require_attr(block, "le_videos"), "le_videos")
    rv = _resolve_eye_video(rv_raw, eye_video_mode)
    lv = _resolve_eye_video(lv_raw, eye_video_mode)

    arena_list = _require_attr(block, "arena_videos")
    arena_path = _choose_arena_video(arena_list, arena_video)

    capR = _open_cap(rv, "right_eye")
    capL = _open_cap(lv, "left_eye")
    capA = _open_cap(arena_path, "arena")

    rR = MonotoneFrameReader(rv, "right_eye")
    rL = MonotoneFrameReader(lv, "left_eye")
    rA = MonotoneFrameReader(arena_path, "arena")

    Wr, Hr = int(capR.get(cv2.CAP_PROP_FRAME_WIDTH)), int(capR.get(cv2.CAP_PROP_FRAME_HEIGHT))
    Wl, Hl = int(capL.get(cv2.CAP_PROP_FRAME_WIDTH)), int(capL.get(cv2.CAP_PROP_FRAME_HEIGHT))
    Wa, Ha = int(capA.get(cv2.CAP_PROP_FRAME_WIDTH)), int(capA.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # -------------------------- output geometry --------------------------
    Heye = int(min(Hr, Hl))
    trace_h_eff = max(1, int(round(float(trace_h) * float(trace_scale))))

    Wr_out = max(1, int(round(Wr * (Heye / float(Hr)))))
    Wl_out = max(1, int(round(Wl * (Heye / float(Hl)))))
    Wa_out = max(1, int(round(Wa * (Heye / float(Ha)))))

    Hrow = Heye
    Wtotal = Wr_out + Wa_out + Wl_out
    Htotal = top_banner_h + Hrow + trace_h_eff

    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    writer = cv2.VideoWriter(
        str(out_path),
        cv2.VideoWriter_fourcc(*codec),
        float(fps),
        (Wtotal, Htotal),
    )
    if not writer.isOpened():
        for cap in (capR, capL, capA):
            cap.release()
        raise RuntimeError(f"Could not open VideoWriter for: {out_path} (codec='{codec}')")

    banner = _make_banner(Wtotal, "Video S1. Synchronized Video Example")

    if show_debug_prints:
        print(f"[export] rows={len(t_ms)} | output={Wtotal}x{Htotal} | fps={fps}")
        print(f"[frames] A[{A_frames.min()}..{A_frames.max()}]  L[{L_frames.min()}..{L_frames.max()}]  R[{R_frames.min()}..{R_frames.max()}]")
        src = "centered" if use_centered_eye_data else "raw"
        print(f"[traces] source={src} | phi='{trace_col_map.get('phi')}' theta='{trace_col_map.get('theta')}'")
        print(f"[sizes] R={Wr}x{Hr} -> {Wr_out}x{Heye} | A={Wa}x{Ha} -> {Wa_out}x{Heye} | L={Wl}x{Hl} -> {Wl_out}x{Heye}")

    prev_R = np.zeros((Hr, Wr, 3), dtype=np.uint8)
    prev_L = np.zeros((Hl, Wl, 3), dtype=np.uint8)
    prev_A = np.zeros((Ha, Wa, 3), dtype=np.uint8)

    try:
        for i in tqdm(range(len(t_ms)), desc="Exporting montage", unit="frame", dynamic_ncols=True):
            tcur = float(t_ms[i])

            idxR = int(R_frames[i]) if R_frames[i] >= 0 else None
            idxL = int(L_frames[i]) if L_frames[i] >= 0 else None
            idxA = int(A_frames[i]) if A_frames[i] >= 0 else None

            idxR = _clamp_idx(idxR, capR)
            idxL = _clamp_idx(idxL, capL)
            idxA = _clamp_idx(idxA, capA)

            missingR = missingL = missingA = False

            fR = rR.read_at(idxR) if idxR is not None else None
            if fR is None:
                fR = prev_R.copy(); missingR = True
            else:
                prev_R = fR.copy()

            fL = rL.read_at(idxL) if idxL is not None else None
            if fL is None:
                fL = prev_L.copy(); missingL = True
            else:
                prev_L = fL.copy()

            fA = rA.read_at(idxA) if idxA is not None else None
            if fA is None:
                fA = prev_A.copy(); missingA = True
            else:
                prev_A = fA.copy()

            if flip_eyes_vertical:
                fR = cv2.flip(fR, 0)
                fL = cv2.flip(fL, 0)

            # overlays
            _safe_put_text(fR, "RIGHT", (12, 24), (255, 255, 255), scale=0.75, thickness=2)
            _safe_put_text(fA, "ARENA", (12, 24), (255, 255, 255), scale=0.75, thickness=2)
            _safe_put_text(fL, "LEFT",  (12, 24), (255, 255, 255), scale=0.75, thickness=2)

            if show_disqualified_badge and disqR_flags[i]:
                _safe_put_text(fR, "disqualified", (max(10, fR.shape[1] - 170), 24), (0, 0, 255), scale=0.65, thickness=2)
            if show_disqualified_badge and disqL_flags[i]:
                _safe_put_text(fL, "disqualified", (max(10, fL.shape[1] - 170), 24), (0, 0, 255), scale=0.65, thickness=2)

            if missingR:
                _safe_put_text(fR, "missing frame", (12, fR.shape[0] - 12), (255, 0, 0), scale=0.6, thickness=2)
            if missingL:
                _safe_put_text(fL, "missing frame", (12, fL.shape[0] - 12), (255, 0, 0), scale=0.6, thickness=2)
            if missingA:
                _safe_put_text(fA, "missing frame", (12, fA.shape[0] - 12), (255, 0, 0), scale=0.6, thickness=2)

            ts_str = f"{tcur:.{timestamp_precision_ms}f} ms"
            ts_sz, _ = cv2.getTextSize(ts_str, cv2.FONT_HERSHEY_SIMPLEX, 0.85, 2)
            tx = max(12, (fA.shape[1] - ts_sz[0]) // 2)
            ty = max(28, fA.shape[0] - 12)
            _safe_put_text(fA, ts_str, (tx, ty), (255, 255, 255), scale=0.85, thickness=2)

            # force all three to same height (eye height reference)
            fR = _resize_to_height(fR, Heye)
            fA = _resize_to_height(fA, Heye)
            fL = _resize_to_height(fL, Heye)

            row_img = np.concatenate([fR, fA, fL], axis=1)

            # pad/crop to exact Wtotal
            if row_img.shape[1] != Wtotal:
                if row_img.shape[1] < Wtotal:
                    row_img = cv2.copyMakeBorder(row_img, 0, 0, 0, Wtotal - row_img.shape[1],
                                                 cv2.BORDER_CONSTANT, value=(0, 0, 0))
                else:
                    row_img = row_img[:, :Wtotal, :]

            trace = _draw_trace_panel(
                W=Wtotal,
                t_ms=tcur,
                t_grid=t_ms,
                Lsig=Lsig,
                Rsig=Rsig,
                t0=float(t_ms[0]),
                t1=float(t_ms[-1]),
                trace_h_eff=trace_h_eff,
            )

            frame = np.zeros((Htotal, Wtotal, 3), dtype=np.uint8)
            frame[0:top_banner_h, :, :] = banner
            frame[top_banner_h:top_banner_h + Hrow, :, :] = row_img
            frame[top_banner_h + Hrow:top_banner_h + Hrow + trace_h_eff, :, :] = trace

            writer.write(frame)

        return out_path

    finally:
        rR.close(); rL.close(); rA.close()
        try:
            writer.release()
        except Exception:
            pass
        for cap in (capR, capL, capA):
            try:
                cap.release()
            except Exception:
                pass


In [ ]:
# create the video here
start_time_seconds = 240
end_time_seconds = 300
export_block_synchronized_montage_video(
    block,
    start_ms=start_time_seconds*1000,
    end_ms=end_time_seconds*1000,
    out_path=Path(block.analysis_path) / f"montage_finalsync_V7{start_time_seconds}_{end_time_seconds}.mp4",
    fps=60.0,
    eye_video_mode="raw",   # or "dlc" / "auto"
    arena_frame_shift=-4,
    require_all_three=True,
)